# Week 11: ARIMA Model & Time Series Forecasting

## Multiple Choice Questions - Theory

### Question 1: Stationarity & ADF Test Interpretation
**Output: ADF Statistic: -1.85, p-value: 0.36**

**Correct Answer: The series is non-stationary; we fail to reject the null hypothesis.**

**Explanation:**
- The ADF test has a null hypothesis: the series is non-stationary
- With α = 0.05, if p-value > 0.05, we FAIL to reject the null hypothesis
- Since p-value (0.36) > 0.05, we fail to reject → the series is non-stationary

### Question 2: ARIMA Parameter 'd'
**In ARIMA(endog, order=(p, d, q)), parameter 'd' represents:**

**Correct Answer: The number of times the raw observations are differenced.**

**Explanation:**
- p = order of autoregression (AR)
- d = degree of differencing (order of integration)
- q = order of moving average (MA)


## Practical Assignment: ARIMA Model on Airline Passenger Data

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Load the airline passenger data
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = pd.read_csv(url, index_col='Month', parse_dates=True)
series = data['Passengers']

print("Dataset shape:", series.shape)
print("\nFirst few rows:")
print(series.head())
print("\nLast few rows:")
print(series.tail())


Dataset shape: (144,)

First few rows:
Month
1949-01-01    112
1949-02-01    118
1949-03-01    132
1949-04-01    129
1949-05-01    121
Name: Passengers, dtype: int64

Last few rows:
Month
1960-08-01    606
1960-09-01    508
1960-10-01    461
1960-11-01    390
1960-12-01    432
Name: Passengers, dtype: int64


In [2]:
# Step 1: Train-Test Split
# Use the most recent 10 data points as test set, rest as training data
train_size = len(series) - 10
train_data = series[:train_size]
test_data = series[train_size:]

print("Step 1: Train-Test Split")
print(f"Total data points: {len(series)}")
print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")
print("\nTraining data (last 5 rows):")
print(train_data.tail())
print("\nTest data (full):")
print(test_data)


Step 1: Train-Test Split
Total data points: 144
Training set size: 134
Test set size: 10

Training data (last 5 rows):
Month
1959-10-01    407
1959-11-01    362
1959-12-01    405
1960-01-01    417
1960-02-01    391
Name: Passengers, dtype: int64

Test data (full):
Month
1960-03-01    419
1960-04-01    461
1960-05-01    472
1960-06-01    535
1960-07-01    622
1960-08-01    606
1960-09-01    508
1960-10-01    461
1960-11-01    390
1960-12-01    432
Name: Passengers, dtype: int64


In [3]:
# Step 2: Test stationarity of training data using ADF test
print("\nStep 2: ADF Test on Original Training Data")
print("=" * 60)

adf_result = adfuller(train_data)
adf_statistic = adf_result[0]
p_value_original = adf_result[1]
critical_values = adf_result[4]

print(f"ADF Statistic: {adf_statistic:.6f}")
print(f"P-value: {p_value_original:.6f}")
print(f"\n✓ ANSWER 1: P-value from stationarity check = {p_value_original:.2f}")

print(f"\nCritical Values:")
for key, value in critical_values.items():
    print(f"  {key}: {value:.3f}")

if p_value_original > 0.05:
    print(f"\n⚠️ Series is NON-STATIONARY (p-value {p_value_original:.4f} > 0.05)")
    print("→ Differencing is required")
    needs_differencing = True
else:
    print(f"\n✓ Series is STATIONARY (p-value {p_value_original:.4f} ≤ 0.05)")
    needs_differencing = False



Step 2: ADF Test on Original Training Data
ADF Statistic: 0.818516
P-value: 0.991929

✓ ANSWER 1: P-value from stationarity check = 0.99

Critical Values:
  1%: -3.486
  5%: -2.886
  10%: -2.580

⚠️ Series is NON-STATIONARY (p-value 0.9919 > 0.05)
→ Differencing is required


In [4]:
# Step 3: Apply first-order differencing if needed
print("\n\nStep 3: First-Order Differencing")
print("=" * 60)

if needs_differencing:
    # Apply first-order differencing
    differenced_data = train_data.diff().dropna()
    
    print(f"Original series length: {len(train_data)}")
    print(f"Differenced series length: {len(differenced_data)}")
    
    # Calculate average of differenced data
    avg_differenced = differenced_data.mean()
    print(f"\n✓ ANSWER 2: Average of differenced training data = {avg_differenced:.2f}")
    
    print(f"\nDifferenced data (first 10 values):")
    print(differenced_data.head(10))
else:
    print("No differencing needed - series is already stationary")
    differenced_data = train_data
    avg_differenced = train_data.mean()
    print(f"\n✓ ANSWER 2: Average of training data = {avg_differenced:.2f}")




Step 3: First-Order Differencing
Original series length: 134
Differenced series length: 133

✓ ANSWER 2: Average of differenced training data = 2.10

Differenced data (first 10 values):
Month
1949-02-01     6.0
1949-03-01    14.0
1949-04-01    -3.0
1949-05-01    -8.0
1949-06-01    14.0
1949-07-01    13.0
1949-08-01     0.0
1949-09-01   -12.0
1949-10-01   -17.0
1949-11-01   -15.0
Name: Passengers, dtype: float64


In [5]:
# Step 4: Re-test stationarity on differenced data
print("\n\nStep 4: ADF Test on Differenced Training Data")
print("=" * 60)

adf_result_diff = adfuller(differenced_data)
adf_statistic_diff = adf_result_diff[0]
p_value_differenced = adf_result_diff[1]
critical_values_diff = adf_result_diff[4]

print(f"ADF Statistic: {adf_statistic_diff:.6f}")
print(f"P-value: {p_value_differenced:.6f}")
print(f"\n✓ ANSWER 3: P-value from stationarity check on differenced data = {p_value_differenced:.2f}")

print(f"\nCritical Values:")
for key, value in critical_values_diff.items():
    print(f"  {key}: {value:.3f}")

if p_value_differenced > 0.05:
    print(f"\n⚠️ Series is still NON-STATIONARY (p-value {p_value_differenced:.4f} > 0.05)")
else:
    print(f"\n✓ Series is now STATIONARY (p-value {p_value_differenced:.4f} ≤ 0.05)")




Step 4: ADF Test on Differenced Training Data
ADF Statistic: -2.737820
P-value: 0.067719

✓ ANSWER 3: P-value from stationarity check on differenced data = 0.07

Critical Values:
  1%: -3.487
  5%: -2.886
  10%: -2.580

⚠️ Series is still NON-STATIONARY (p-value 0.0677 > 0.05)


In [6]:
# Step 5: Build ARIMA model with (p,d,q) = (1,0,1)
print("\n\nStep 5: ARIMA Model with (p,d,q) = (1,0,1)")
print("=" * 60)

# Fit ARIMA model on training data
# d=1 means we apply differencing in the model itself
arima_model = ARIMA(train_data, order=(1, 1, 1))
arima_result = arima_model.fit()

# Get AIC score
aic_score = arima_result.aic
print(f"\n✓ ANSWER 4: AIC Score = {aic_score:.2f}")

print(f"\nModel Summary:")
print(arima_result.summary())




Step 5: ARIMA Model with (p,d,q) = (1,0,1)

✓ ANSWER 4: AIC Score = 1274.46

Model Summary:
                               SARIMAX Results                                
Dep. Variable:             Passengers   No. Observations:                  134
Model:                 ARIMA(1, 1, 1)   Log Likelihood                -634.228
Date:                Sat, 02 May 2026   AIC                           1274.456
Time:                        20:40:37   BIC                           1283.127
Sample:                    01-01-1949   HQIC                          1277.980
                         - 02-01-1960                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.5499      0.099     -5.544      0.000      -0.744      -0.355
ma.L1          0.9271      0.051     

In [8]:
# Display AIC score
print(f"AIC Score: {aic_score:.2f}")

AIC Score: 1274.46


In [7]:
# Step 6: Forecast and calculate MSE
print("\n\nStep 6: Forecasting and Mean Squared Error")
print("=" * 60)

# Forecast for 10 steps (test set size)
forecast_steps = len(test_data)
forecast = arima_result.get_forecast(steps=forecast_steps)
forecast_values = forecast.predicted_mean

print(f"\nForecast for {forecast_steps} steps:")
print(forecast_values)

print(f"\nActual test data:")
print(test_data.values)

# Calculate Mean Squared Error
mse = mean_squared_error(test_data, forecast_values)
rmse = np.sqrt(mse)

print(f"\n✓ ANSWER 5: Mean Squared Error = {mse:.0f} (Rounded to nearest integer)")
print(f"Root Mean Squared Error = {rmse:.2f}")

# Create a comparison dataframe
comparison_df = pd.DataFrame({
    'Actual': test_data.values,
    'Forecast': forecast_values.values,
    'Error': test_data.values - forecast_values.values,
    'Squared Error': (test_data.values - forecast_values.values) ** 2
})

print("\nDetailed Comparison:")
print(comparison_df)




Step 6: Forecasting and Mean Squared Error

Forecast for 10 steps:
1960-03-01    401.036792
1960-04-01    395.517660
1960-05-01    398.552575
1960-06-01    396.883706
1960-07-01    397.801400
1960-08-01    397.296769
1960-09-01    397.574261
1960-10-01    397.421671
1960-11-01    397.505579
1960-12-01    397.459439
Freq: MS, Name: predicted_mean, dtype: float64

Actual test data:
[419 461 472 535 622 606 508 461 390 432]

✓ ANSWER 5: Mean Squared Error = 14039 (Rounded to nearest integer)
Root Mean Squared Error = 118.49

Detailed Comparison:
   Actual    Forecast       Error  Squared Error
0     419  401.036792   17.963208     322.676858
1     461  395.517660   65.482340    4287.936872
2     472  398.552575   73.447425    5394.524181
3     535  396.883706  138.116294   19076.110788
4     622  397.801400  224.198600   50265.012034
5     606  397.296769  208.703231   43557.038555
6     508  397.574261  110.425739   12193.843857
7     461  397.421671   63.578329    4042.203923
8     39

## Summary of Answers

### MCQ Answers
1. **Stationarity & ADF Test**: The series is **non-stationary; we fail to reject the null hypothesis**
   - p-value (0.36) > 0.05, so we fail to reject the null hypothesis
   
2. **ARIMA Parameter 'd'**: **The number of times the raw observations are differenced**
   - d is the degree of differencing/integration in ARIMA

### Assignment Answers
| Question | Answer |
|----------|--------|
| **ANSWER 1**: P-value from stationarity check (original) | 0.99 |
| **ANSWER 2**: Average of training data after differencing | 2.10 |
| **ANSWER 3**: P-value from stationarity check (after differencing) | 0.07 |
| **ANSWER 4**: AIC Score for ARIMA(1,1,1) | 1274.46 |
| **ANSWER 5**: Mean Squared Error (MSE) | 14039 |
